# Spark 🤗 Hugging Face{background-color="black"}

*Run (almost) any open model on your Spark DataFrames*

- **Hub**: ~2M models, ~500k datasets, Spaces — the "GitHub of AI"
- **Libraries**: `transformers`, `tokenizers`, `datasets`, `safetensors`, `optimum`, `huggingface_hub`
- Tasks: text classification, NER, QA, summarization, translation, embeddings, ASR, vision, LLMs
- In Spark we use the model **inside Python workers** via Arrow-based **Pandas UDFs**

## The Hub in 5 minutes

:::: {.columns}

::: {.fragment .column width="50%"}
**A model repo** = git repo with

- `config.json` (architecture, labels)
- weights in **safetensors** (safe, memory-mapped) or GGUF/ONNX variants
- tokenizer files
- **model card** (`README.md`): intended use, data, license, limitations

Always check the **license** and the **model card** before using a model in a project.
:::

::: {.fragment .column width="50%"}
```python
from transformers import pipeline

clf = pipeline("sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    device=-1)                     # -1 = CPU

clf(["This lesson is not boring at all",
     "I hate waiting for the bus"])
# [{'label': 'POSITIVE', 'score': 0.99...},
#  {'label': 'NEGATIVE', 'score': 0.99...}]
```

Models are cached in `HF_HOME` (default `~/.cache/huggingface`): mount it as a **volume**.
:::
::::

## Setup with Docker

:::: {.columns}
::: {.column width="55%"}
`huggingface/Dockerfile`
```Dockerfile
FROM apache/spark:4.3.0

USER root

# CPU-only PyTorch keeps the image ~2 GB instead of ~8 GB
RUN pip install --no-cache-dir \
    "pandas<3" pyarrow transformers \
    "optimum[onnxruntime]" \
    torch --extra-index-url https://download.pytorch.org/whl/cpu

RUN mkdir -p /opt/spark/.cache/huggingface && \
    chown -R spark:spark /opt/spark/.cache
ENV HF_HOME=/opt/spark/.cache/huggingface

USER spark
```
:::

::: {.column width="45%"}
```bash
cd huggingface
docker build -t tap:sparkhf .

docker run --rm -it \
  -v ./src:/opt/spark/work-dir \
  -v huggingface-cache:/opt/spark/.cache/huggingface \
  tap:sparkhf \
  /opt/spark/bin/spark-submit sentiment.py
```
:::
::::

## The wrong way: a row-at-a-time UDF

```python
from pyspark.sql.functions import udf

@udf("string")
def sentiment(text):
    clf = pipeline("sentiment-analysis", model=MODEL)   # loads 260 MB... for every row!
    return clf(text)[0]["label"]
```

::: {.fragment}
Three problems:

1. the model is **loaded for every row** (even moving it outside the function, it is pickled and shipped with each task)
2. rows are serialized **one by one** between JVM and Python (pickle)
3. no **batching** → the neural network sees batch size 1
:::

## Pandas UDF + lazy model per worker

Arrow moves **columnar batches** (default 10,000 rows) from JVM to Python; the model is loaded **once per Python worker**.

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf
from transformers import pipeline

MODEL = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
_clf = None                                   # one instance per Python worker

def get_clf():
    global _clf
    if _clf is None:
        _clf = pipeline("sentiment-analysis", model=MODEL, device=-1)
    return _clf

@pandas_udf("label string, score double")     # returns a struct
def hf_sentiment(texts: pd.Series) -> pd.DataFrame:
    preds = get_clf()(texts.tolist(), batch_size=32, truncation=True)
    return pd.DataFrame(preds)

spark = SparkSession.builder.appName("Sentiment HF").getOrCreate()
df = spark.createDataFrame([("This lesson is not boring at all",),
                            ("I hate waiting for the bus",)], ["text"])
df.withColumn("sentiment", hf_sentiment("text")).select("text", "sentiment.*").show(truncate=False)

## Iterator Pandas UDF: load once per task, explicitly

`Iterator[pd.Series] -> Iterator[pd.Series]`: the setup code runs **once**, then all the batches of the partition stream through.

In [ ]:
from typing import Iterator

@pandas_udf("label string, score double")
def hf_sentiment_iter(batches: Iterator[pd.Series]) -> Iterator[pd.DataFrame]:
    clf = pipeline("sentiment-analysis", model=MODEL, device=-1)   # setup: once per task
    for texts in batches:
        yield pd.DataFrame(clf(texts.tolist(), batch_size=32, truncation=True))

df.select("text", hf_sentiment_iter("text").alias("s")).select("text", "s.*").show(truncate=False)

::: {.fragment}
The same pattern works with `mapInPandas` when you want to emit **any number of rows/columns** per input batch (e.g. NER → one row per entity, or chunking documents for RAG).
:::

## `predict_batch_udf`: the official helper

Since Spark 3.4, `pyspark.ml.functions.predict_batch_udf` wraps the iterator pattern: model caching per worker, fixed-size batches, NumPy in/out.

In [ ]:
import numpy as np
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

def make_predict_fn():
    clf = pipeline("sentiment-analysis", model=MODEL, device=-1)   # called once per worker
    def predict(texts: np.ndarray) -> dict:
        out = clf(texts.tolist(), truncation=True)
        return {"label": np.array([o["label"] for o in out]),
                "score": np.array([o["score"] for o in out])}
    return predict

sentiment_udf = predict_batch_udf(
    make_predict_fn,
    return_type=StructType([StructField("label", StringType()), StructField("score", DoubleType())]),
    batch_size=64)

df.withColumn("s", sentiment_udf("text")).select("text", "s.*").show(truncate=False)

## Faster on CPU: the same model in ONNX

From the [Spark NLP](SparkNLP.ipynb) notebook: export once with Optimum, run with ONNX Runtime — the UDF does not change.

```python
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer, pipeline

def make_clf():
    model = ORTModelForSequenceClassification.from_pretrained(MODEL, export=True)  # or a local ONNX dir
    tok = AutoTokenizer.from_pretrained(MODEL)
    return pipeline("sentiment-analysis", model=model, tokenizer=tok)
```

::: {.fragment}
Typical gain on CPU: **1.5-3x** (more with INT8 quantization). PyTorch is still needed only if you export at runtime: export at **build time** to get a lighter image.
:::

## Applying it to a stream

The UDF is just a column expression, so it works unchanged in **Structured Streaming**:

```python
tweets = (spark.readStream.format("kafka")
          .option("kafka.bootstrap.servers", "broker:9092")
          .option("subscribe", "tweets").load()
          .selectExpr("CAST(value AS STRING) AS text"))

(tweets.withColumn("sentiment", hf_sentiment("text"))
       .writeStream.format("parquet")
       .option("path", "/data/tweets_sentiment")
       .option("checkpointLocation", "/data/chk/tweets_sentiment")
       .start())
```

::: {.fragment}
⚠️ Each micro-batch pays the inference cost: watch the **processing time vs trigger interval** in the Spark UI (port 4040).
:::

## Performance checklist

| Knob | Why |
|---|---|
| `spark.sql.execution.arrow.maxRecordsPerBatch` | rows per Arrow batch sent to Python (default 10,000) |
| model `batch_size` | rows per forward pass; tune with memory |
| `torch.set_num_threads(n)` | avoid **oversubscription**: executor cores × torch threads ≤ physical cores |
| `repartition(n)` | enough partitions to use all executors, not so many that each reloads the model |
| volume on `HF_HOME` | download once, not at each container start |
| `spark.task.resource.gpu.amount` | schedule GPU tasks explicitly on GPU clusters |

## The limits of this approach

- Fine for **encoders** (classification, NER, embeddings) up to a few hundred MB
- With an **LLM** each Python worker would load **GBs** of weights and generate token by token on CPU
- Model and data pipeline scale **together**: you cannot add GPUs without adding executors

::: {.fragment}
Keep the batching pattern, **move the model out**: inside `mapInPandas` we will call an **inference server** with an OpenAI-compatible API.

```python
def enrich(batches):
    client = OpenAI(base_url="http://ollama:11434/v1", api_key="none")   # once per task
    for pdf in batches:
        pdf["summary"] = [ask(client, t) for t in pdf.text]                # concurrent requests
        yield pdf
```

👉 **[Local Inference: llama.cpp, Ollama, vLLM](Local%20Inference.ipynb)**
:::

## Exercise

1. Use `hf_sentiment_iter` on 10,000 Sentiment140 tweets; measure time with `batch_size` 1, 8, 32, 128
2. Swap the model with the ONNX version: what changes in time and image size?
3. Use `mapInPandas` with a NER model (`dslim/bert-base-NER`, `aggregation_strategy="simple"`) and emit one row per entity

# Biblio

* https://huggingface.co/docs/transformers/main_classes/pipelines
* https://huggingface.co/docs/hub/models-cards
* https://spark.apache.org/docs/latest/api/python/user_guide/sql/arrow_pandas.html
* https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.functions.predict_batch_udf.html
* https://huggingface.co/docs/optimum-onnx/
* https://www.databricks.com/blog/2023/02/06/getting-started-nlp-using-hugging-face-transformers-pipelines.html